In [12]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import math
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [14]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

In [15]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=256, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=256, shuffle=True)

In [16]:
torch.manual_seed(1234)

In [17]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        sm = F.log_softmax(logits)
        return sm

model = NeuralNetwork()


In [18]:
model.to(device)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [19]:
learning_rate = 1e-3
epochs = 500

In [20]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

In [21]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [22]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.309407  [    0/60000]


/tmp/ipykernel_491639/1934186528.py:16: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  sm = F.log_softmax(logits)


loss: 0.602984  [25600/60000]
loss: 0.476927  [51200/60000]
Test Error: 
 Accuracy: 79.9%, Avg loss: 0.546940 

Epoch 2
-------------------------------
loss: 0.444015  [    0/60000]
loss: 0.434224  [25600/60000]
loss: 0.459812  [51200/60000]
Test Error: 
 Accuracy: 84.2%, Avg loss: 0.428256 

Epoch 3
-------------------------------
loss: 0.404380  [    0/60000]
loss: 0.376887  [25600/60000]
loss: 0.345952  [51200/60000]
Test Error: 
 Accuracy: 84.6%, Avg loss: 0.426271 

Epoch 4
-------------------------------
loss: 0.458052  [    0/60000]
loss: 0.361159  [25600/60000]
loss: 0.349736  [51200/60000]
Test Error: 
 Accuracy: 85.6%, Avg loss: 0.402713 

Epoch 5
-------------------------------
loss: 0.344463  [    0/60000]
loss: 0.266368  [25600/60000]
loss: 0.314167  [51200/60000]
Test Error: 
 Accuracy: 83.2%, Avg loss: 0.470833 

Epoch 6
-------------------------------
loss: 0.348100  [    0/60000]
loss: 0.328253  [25600/60000]
loss: 0.347133  [51200/60000]
Test Error: 
 Accuracy: 86.1%,